# Using LLMs and Knowledge graphs to search for PFAS Alternatives

## Project with Saint Gobain

#### Yu-Chuan (Michael) Hsu, Isabella Stewart, Tarjei Hage, Wei Lu, and Markus J. Buehler, MIT, 2025 
mkychsu@MIT.EDU, istewart@MIT.EDU, tphage@MIT.EDU, wl7@MIT.EDU, mbuehler@MIT.EDU
#### LAMM, Massachusetts Institute of Technology


# Allows for distributed or parallel processing of a dataset

In [2]:
import sys, os

try:
    thread_i = int(sys.argv[1]) #This job’s index (e.g., 0, 1, 2, 3, 4)
    total_threads = int(sys.argv[2]) #Total number of jobs (e.g., 5)

except: #If you don’t provide any arguments, it defaults to thread_i=0 and total_threads=1 — meaning it will process everything by itself.
    thread_i = 0 
    total_threads = 1

In [3]:
config_list = [
    {
        "model":"meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
        # "model":"meta-llama/Llama-3.3-70B-Instruct-Turbo",
        # "model":"meta-llama/Llama-3.3-70B-Instruct-Turbo-Free",
        
        "api_key":os.getenv("Llama4_together"),
        "max_tokens": 20000
    },
]

# Client Initiation with Together

In [4]:
from together import Together
client = Together(api_key=config_list[0]["api_key"])


In [5]:
import os
from GraphReasoning import *


In [6]:
verbatim=False

In [7]:
doc_data_dir = './paper' #soft link or absolute path to markdown files
data_dir='./GRAPHDATA_paper'    
data_dir_output='./GRAPHDATA_OUTPUT_paper'

max_tokens = config_list[0]['max_tokens']

embedding_file='SG_paper_LLAMA4_70b.pkl'


# Embedding the graph with Nomic

In [8]:
if total_threads == 1: ##merging mode: This block only runs if you're not distributing work across multiple threads.

  
    from transformers import AutoModelForCausalLM, AutoTokenizer
    # from tqdm.notebook import tqdm
    # from IPython.display import display, Markdown
    
    
    # tokenizer_model=f'/home/mkychsu/pool/llm/SEMIKONG-8b-GPTQ'
    tokenizer_model=f'/home/mkychsu/pool/llm/nomic-embed-text-v1.5'
    
    # embedding_tokenizer = AutoTokenizer.from_pretrained(tokenizer_model,use_fast=False, device_map="cuda:0")
    # embedding_model = AutoModelForCausalLM.from_pretrained(tokenizer_model,output_hidden_states=True).to('cuda:0')
    
    from sentence_transformers import SentenceTransformer
    embedding_tokenizer =''
    embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
    
    
    from GraphReasoning import load_embeddings, save_embeddings, generate_node_embeddings
    
    # generate_new_embeddings=True
    
    # from PIL import Image
    # from transformers import AutoModelForCausalLM 
    # from transformers import AutoProcessor 
    
    # model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda:1", trust_remote_code=True, torch_dtype="auto")
    # processor = AutoProcessor.from_pretrained(model_id, device_map="cuda:1", trust_remote_code=True) 


    import torch
    generate_new_embeddings=True

#This block checks if the embedding file already exists… but then immediately overrides the flag to True anyway, which forces re-generation.
    if os.path.exists(f'{data_dir}/{embedding_file}'):
        print('Found existing embedding file')
        generate_new_embeddings=False
    generate_new_embeddings=True

    #iterates over graph nodes and embeds them using the model 
    with torch.no_grad():
        if generate_new_embeddings:
            G=nx.DiGraph()
            node_embeddings = generate_node_embeddings(G, embedding_tokenizer, embedding_model, )
            save_embeddings(node_embeddings, f'{data_dir}/{embedding_file}')
        else:
            node_embeddings = load_embeddings(f'{data_dir}/{embedding_file}')
            
            


/home/mkychsu/pool/.conda/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
<All keys matched successfully>


Found existing embedding file


0it [00:00, ?it/s]


### Load dataset

In [9]:
### Load dataset of papers

import pandas as pd
import glob

### Load all markdown files (dataset of papers) --- assumes each subfolder in doc_data_dir is named like a paper ID
#doc_list becomes a list of full file paths to .md files
doc_list=[]
with os.scandir(f'{doc_data_dir}') as folders:
    for folder in folders:
        doc_list.append(f'{doc_data_dir}/{folder.name}/{folder.name}.md')

doc_list=sorted(doc_list)


### Set up LLM client:

In [10]:

import instructor
from typing import List
from PIL import Image
import base64

from pydantic import BaseModel

class Node(BaseModel): 
    id: str
    type: str
        
class Edge(BaseModel):
    source: str
    target: str
    relation: str
        
class KnowledgeGraph(BaseModel):
    nodes: List[Node]
    edges: List[Edge]

#Identifies phrases or terms that are potential nodes --> Decide the type of each node --> Determine which nodes are related, and what the relation is 

response_model = KnowledgeGraph
system_prompt = '''
You are a scientific assistant extracting knowledge graphs from text.
Return a JSON with two fields: <nodes> and <edges>.\n
Each node must have <id> and <type>.\n
Each edge must have <source>, <target>, and <relation>.
'''


def generate(system_prompt=system_prompt, 
             prompt="",temperature=0.333,
             max_tokens=config_list[0]['max_tokens'], response_model=KnowledgeGraph, 
            ):     

    if system_prompt==None:
        messages=[
            {"role": "user", "content": f"{prompt}"},
        ]

    else:
        messages=[
            {"role": "system",  "content": f"{system_prompt}"},
            {"role": "user", "content": f"{prompt}"},
        ]

    # monkey patching: replacing or enhancing client.chat.completions.create
    create = instructor.patch(     #instructor is a python library that wraps LLM responses and validates LLM output against a schema you define (via pydantic)
        create=client.chat.completions.create,    #Automatically converts the response into a real Python object (not just a raw string or dictionary)
        mode=instructor.Mode.JSON_SCHEMA,
    ) 
    

    return create(messages=messages,   
                    model=config_list[0]["model"],
                    max_tokens=max_tokens,
                    temperature=0.333,
                    response_model=response_model,
                   )

def image_to_base64_data_uri(file_path):
    with open(file_path, "rb") as image_file:
        base64_data = base64.b64encode(image_file.read()).decode("utf-8")
        return f"data:image/png;base64,{base64_data}"

def generate_figure(image, system_prompt=system_prompt, 
                prompt="", temperature=0,
                ):

    pwd = os.getcwd()
    image = image.split(pwd)[-1]
    image=Path('.').glob(f'**/{image}', case_sensitive=False)
    image = list(image)[0]

    image_uri = image_to_base64_data_uri(image)
    messages = [
        {"role": "system", "content": "You are an assistant who perfectly describes images."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_uri}},
                {"type": "text", "text": "Describe this image in detail please."},
            ],
        },
    ]
        
    return create(messages=messages,   
                    model=config_list[0]["model"],
                    max_tokens=max_tokens,
                    temperature=0.333,
                    response_model=response_model,
                   ).choices[0].message.content

In [11]:
print(f'running on {thread_i}-th thread in totally {total_threads} threads') #double check threading 

running on 0-th thread in totally 1 threads


### Checpoint Cell: Finds where merging left off with current_merged_i. Handles resuming graph merging in single-threaded mode

#### 1. Finds the latest merged graph from previous runs.

#### 2. Checks if it's valid by trying to load it.

#### 3. If the graph or its parts are corrupted, it deletes them and rolls back to the previous merge index.

#### 4. Sets current_merged_i so merging can resume cleanly.

In [30]:
import glob
if total_threads == 1: # merging mode
    # try:
    merged_graph_list = sorted(glob.glob(f'{data_dir_output}/*_integrated.graphml'), reverse=True, key = lambda x: int(x.split('_')[2].split('/')[-1]))
    current_merged_i = int(merged_graph_list[0].split('_')[2].split('/')[-1])
    last_graph = sorted(glob.glob(f'{data_dir}/{current_merged_i}_*.graphml') )
    
    # trim the working graphs from the last session if it is be broken.
    try: 
        nx.read_graphml(merged_graph_list[0])
        nx.read_graphml(last_graph)
    except:
        if os.path.exists(merged_graph_list[0]):
            os.remove(merged_graph_list[0])
        last_graph_files = sorted(glob.glob(f'{data_dir}/{current_merged_i}_*'), reverse=True)
  
        for file in last_graph_files:
            if os.path.exists(file):
                os.remove(file)
        current_merged_i -= 1

    print(f'Start merging from No. {current_merged_i}')
else:
    current_merged_i = 0

Start merging from No. 1683


### Generate a Knowledge Graph (KG) from each document using an LLM + embedding pipeline, then (optionally) merge it into a global graph in merging mode.

In [ ]:

import networkx as nx
from GraphReasoning import make_graph_from_text
from datetime import datetime
import time
import torch
import shutil

G=nx.DiGraph()
with torch.no_grad():
    for i, doc in enumerate(doc_list): #Loop over all the Markdown files (converted from papers) to process each paper one at a time.
        if i % total_threads != thread_i: #Only process documents this thread is responsible for.
            continue
        if i < current_merged_i: #Skip if already processed (current_merged_i was loaded earlier).
            continue

        #extract TITLE, DOI, and the text from the document
        title = doc.split('/')[-1].split('.md')[0] 
        doi = title
        f = open(doc, "r")
        txt = f.read()
        # txt = row['Abstract']

        #Defines where this document’s individual KG will be saved.

        graph_root = f'{i}_{title[:min(100, len(title))]}'
        current_graph = f'{data_dir}/{graph_root}.graphml'
        # image_list = glob.glob(''.join(doc.split('/')[:-1])+'/*png') # if running for image node

        #If the KG .graphml for this doc doesn't exist yet, it runs make_graph_from_text() to generate it
        #uses: The generate() function (LLM extraction), saves the file. 
        while not os.path.exists(current_graph):
            print(f"Generating KG for {i}: {title}")
            try:
                if type(txt) is not str:
                    print(txt)
                    break # format of abstract is wrong 
                now = datetime.now()
                _, current_graph, _, _, _ = make_graph_from_text(txt,generate, #make_graph_from_text returns five values, but we only care about the second one current_graph which is saved to .graphml 
                                      generate_figure, image_list='',
                                      graph_root=graph_root,do_distill=False, #graph_root is A base name (like "5_MyPaperTitle") used to identify and name the .graphml file, do_distill is a summary of text before extracting
                                      chunk_size=10000,chunk_overlap=0, # If the text is very long, it's split into chunks of max 10,000 characters.
                                      repeat_refine=0,verbatim=False, #verbatim is false meaning don't include the raw text in the output files saved alongside the graph 
                                      data_dir=data_dir, #output directory             
                                      save_PDF=False, #no report 
                                     ) # make_graph_from_text is the pipeline function -- generate is the LLM interface function. 
                print("Time: ", datetime.now()-now)
            # successfully generate KGs without error
            except:
                print('Reach rate limit')
                time.sleep(60)

#MERGING MODE: merge newly made graphs to big graph G 
        if total_threads == 1: # merging mode
            if i < current_merged_i: #Skip if this graph index i has already been merged.
                continue
            if i % 100 == 0: #Every 100th paper simplification of the graph is enabled (merge redundant nodes, 
                do_simplify_graph = True #
                size_threshold = 10 #remove disconnected componets  
            else: #otherwise turn off simplification 
                do_simplify_graph = False
                size_threshold = 0
                
            _graph_GraphML= f'{data_dir_output}/{graph_root}_integrated.graphml' 
            if os.path.exists(_graph_GraphML): 
                G = nx.read_graphml(_graph_GraphML)
                print(f'Main KG loaded: {_graph_GraphML}')
                print(f'{G}')
                continue
            
            now = datetime.now()
            print(f'Merging graph No. {i}: {title} to the main one')
            try:
                current_graph = nx.read_graphml(current_graph)
                nx.set_edge_attributes(current_graph, doi, "DOI")
            except:
                continue
            
            _, G, _, node_embeddings, _ = add_new_subgraph_from_text(txt='',
                               node_embeddings=node_embeddings,
                               tokenizer=embedding_tokenizer,
                               model=embedding_model,
                               original_graph=G, data_dir_output=data_dir_output, graph_root=graph_root,
                               do_simplify_graph=do_simplify_graph, size_threshold=size_threshold,
                               do_update_node_embeddings=do_simplify_graph, #Recompute embeddings during merging
                               repeat_refine=0,similarity_threshold=0.9, #Merge nodes with cosine similarity ≥ 0.9
                               do_Louvain_on_new_graph=do_simplify_graph, #Run community detection (e.g., cluster pruning)
                               #whether or not to simplify, uses similiraty_threshold defined above
                               return_only_giant_component=False,
                               save_common_graph=False,G_to_add=current_graph,graph_GraphML_to_add=None,
                               verbatim=True,)
            save_embeddings(node_embeddings, f'{data_dir}/{embedding_file}')
            print("Time: ", datetime.now()-now)

Main KG loaded: ./GRAPHDATA_OUTPUT_paper/1683_10.1016_j.coche.2023.100943_integrated.graphml
DiGraph with 67356 nodes and 183400 edges
Main KG loaded: ./GRAPHDATA_OUTPUT_paper/1684_10.1016_j.coche.2023.100947_integrated.graphml
DiGraph with 67383 nodes and 183444 edges
Main KG loaded: ./GRAPHDATA_OUTPUT_paper/1685_10.1016_j.coche.2024.101017_integrated.graphml
DiGraph with 67410 nodes and 183491 edges
Merging graph No. 1686: 10.1016_j.coche.2024.101018 to the main one
Merging graph No. 1687: 10.1016_j.coesh.2024.100588 to the main one


...

Now create or load new graph...
--- 4.744529724121094e-05 seconds ---
Now grow the existing graph...
DiGraph with 67410 nodes and 183491 edges DiGraph with 94 nodes and 70 edges
Done update graph
Save new graph as:  ./GRAPHDATA_OUTPUT_paper/1687_10.1016_j.coesh.2024.100588_integrated.graphml
Done saving new graph
DiGraph with 67464 nodes and 183555 edges ./GRAPHDATA_OUTPUT_paper/1687_10.1016_j.coesh.2024.100588_integrated.graphml
Time:  0:00:11.417648
Merging graph No. 1688: 10.1016_j.cogsc.2023.100775 to the main one


...

Now create or load new graph...
--- 4.8160552978515625e-05 seconds ---
Now grow the existing graph...
DiGraph with 67464 nodes and 183555 edges DiGraph with 116 nodes and 99 edges
Done update graph
Save new graph as:  ./GRAPHDATA_OUTPUT_paper/1688_10.1016_j.cogsc.2023.100775_integrated.graphml
Done saving new graph
DiGraph with 67531 nodes and 183628 edges ./GRAPHDATA_OUTPUT_paper/1688_10.1016_j.cogsc.2023.100775_integrated.graphml
Time:  0:00:09.764231
Merging graph No. 1689: 10.1016_j.cogsc.2024.100960 to the main one


...

Now create or load new graph...
--- 4.5299530029296875e-05 seconds ---
Now grow the existing graph...
DiGraph with 68246 nodes and 184645 edges DiGraph with 69 nodes and 54 edges
Done update graph
Save new graph as:  ./GRAPHDATA_OUTPUT_paper/1698_10.1016_j.ebiom.2024.105422_integrated.graphml
Done saving new graph
DiGraph with 68296 nodes and 184698 edges ./GRAPHDATA_OUTPUT_paper/1698_10.1016_j.ebiom.2024.105422_integrated.graphml
Time:  0:00:09.632647
Merging graph No. 1699: 10.1016_j.ecoenv.2020.111805 to the main one


...

Now create or load new graph...
--- 4.553794860839844e-05 seconds ---
Now grow the existing graph...
DiGraph with 68296 nodes and 184698 edges DiGraph with 125 nodes and 144 edges
Done update graph
Save new graph as:  ./GRAPHDATA_OUTPUT_paper/1699_10.1016_j.ecoenv.2020.111805_integrated.graphml
Done saving new graph
DiGraph with 68343 nodes and 184807 edges ./GRAPHDATA_OUTPUT_paper/1699_10.1016_j.ecoenv.2020.111805_integrated.graphml
Time:  0:00:12.388791
Merging graph No. 1700: 10.1016_j.ecoenv.2021.112902 to the main one


...

Now create or load new graph...
--- 5.221366882324219e-05 seconds ---
Now grow the existing graph...
DiGraph with 68343 nodes and 184807 edges DiGraph with 100 nodes and 89 edges
Now update node embeddings
Adding embedding for node: ion pair cleanup
Adding embedding for node: Cd2+ detection
Adding embedding for node: TGA-MS
Adding embedding for node: pentafluorobenzoic acid
Adding embedding for node: x:3 Uacid
Adding embedding for node: fluoroplastics
Adding embedding for node: U.S. Fish and Wildlife Service
Adding embedding for node: Antioxidant levels
Adding embedding for node: Grand Canyon
Adding embedding for node: human studies
Adding embedding for node: Pogonatherum crinitum
Adding embedding for node: Precipitation data
Adding embedding for node: Chloroacetic acid
Adding embedding for node: Drainage water
Adding embedding for node: helophytes
Adding embedding for node: American oysters
Adding embedding for node: Lopez-Espinosa et al. (2016)
Adding embedding for node: Bimetallic i

### Test